In [1]:
"""
exp01 Step 02: Single-Feature Signal Check (stopping rule, PROTOCOL.md §13.1; base-rate version since protocol 1.0)

Question: does any single feature move the take profit rate AWAY FROM THE BASE RATE (the unconditional TP rate of the same
window at the same SL/TP distance), reliably, on training AND on the pooled validation quarters?

Changed from the v1 check (signal_check_v1, 2026-10-01; see docs/history): v1 compared bins with the BREAK-EVEN TP rate,
and market drift alone lifted 1,339 training bins above it. Now:
    - reference = the window's base rate per delta (two-sided: below the base rate = "do not enter here");
    - validation = the 4 most recent validation quarters POOLED; training = the 10-year window of the first pooled fold;
    - a bin's pass counts only if its rows cover at least 6 calendar months in each window;
    - the break-even TP rate is still reported (an above-base-rate bin can still be untradable after costs).

Per (feature, delta, bin): TP rate with a block-bootstrap interval (whole ISO weeks resampled), base rate, break-even rate,
mean net return, month count. The statistic is the number of DISTINCT FEATURE BINS with at least one signal.

Calibration (added 2026-10-05; the first real run of this notebook did not have it): with ~3,000 correlated tests, signals appear by chance (a
synthetic random walk produced 2). So the same check is also run 19 times on a NULL: the feature columns shifted together
by a random number of whole sessions (25%-75% of each window) against the unchanged outcomes, which breaks any
feature -> outcome link but keeps everything else. Verdict: CONTINUE only if the real check has more distinct signal bins
than EVERY null run (p <= 0.05; ties stop), otherwise STOP or pivot. The null runs use all CPU cores (joblib) and take
about 19 times one check divided by the number of cores. One signal-check entry is written to the trial log.
Prerequisite: exp01 step 01. Outputs: store04_experiments/exp01_minute_entry/step02_signal_check_data/.
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import time
import pandas as pd
import numpy as np
# IMPORT PLOTLY
import plotly.express as px
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp01_minute_entry import config as exp_config
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import get_date_range_file_path_list, write_csv_file_to_path
# IMPORT BARRIER LABEL AND TRADE EXECUTION FUNCTIONS
from so.features.barrier_labels import get_delta_col_str
from so.core.trade_execution import get_breakeven_tp_rate_arr
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp01_minute_entry.model_dataset import read_model_dataset_pdf, get_model_feature_col_str_list
from experiments.exp01_minute_entry.signal_check import get_signal_check_pdf, get_signal_check_verdict_dict, get_null_signal_count_pdf
from experiments.exp01_minute_entry.walk_forward import get_fold_pdf
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

ImportError: cannot import name 'get_null_signal_count_pdf' from 'experiments.exp01_minute_entry.signal_check' (C:\Users\nico\Desktop\stock_overflow_workspace\experiments\exp01_minute_entry\signal_check.py)

In [ ]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step02_signal_check_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "signal_check")

In [ ]:
# DEFINE WHETHER ABSOLUTE-DOLLAR FEATURES ARE INCLUDED (THEIR BINS MOSTLY SEPARATE YEARS / PRICE LEVELS UNTIL SCALING IS DESIGNED)
INCLUDE_ABSOLUTE_FEATURES_BOOL = False
# COLLECT THE SESSION DATES FROM THE MODEL DATASET FILES
dataset_file_path_list = get_date_range_file_path_list(paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step01_model_dataset_data"))
session_date_list = [pd.to_datetime(f.split("_")[-1].replace(".csv", "")).date() for f in dataset_file_path_list]
# CALL FUNCTION TO GET THE FOLDS (SIGNAL CHECK TRAINING WINDOW ONLY) AND COLLECT THE POOLED FOLDS
fold_pdf = get_fold_pdf(session_date_list, train_window_years_list_in=[exp_config.SIGNAL_CHECK_TRAIN_WINDOW_YEARS])
pooled_fold_pdf = fold_pdf.iloc[-exp_config.SIGNAL_CHECK_POOLED_QUARTER_COUNT:]
first_pooled_fold_row = pooled_fold_pdf.iloc[0]
# DEFINE THE TRAINING WINDOW (10 YEARS, ENDING AN EMBARGO BEFORE THE FIRST POOLED VALIDATION QUARTER)
train_start, train_end = first_pooled_fold_row[f"train_start_{exp_config.SIGNAL_CHECK_TRAIN_WINDOW_YEARS}y"], first_pooled_fold_row["train_end"]
# DISPLAY INFORMATION
print(f"Train:\t\t{train_start} -> {train_end}")
for _, row in pooled_fold_pdf.iterrows():
    print(f"Validation:\t{row['valid_start']} -> {row['valid_end']}\t(fold {row['fold_id']})")
print(f"Latest test window (not read):\t{fold_pdf['test_start'].iloc[-1]} -> {fold_pdf['test_end'].iloc[-1]}")

In [ ]:
# CALL FUNCTION TO READ THE TRAINING ROWS (SAMPLED AS FOR TRAINING, TO LIMIT OVERLAPPING TRADES)
signal_train_pdf = read_model_dataset_pdf(str(train_start), str(train_end), exp_config.SIGNAL_CHECK_DELTA_LIST, exp_config.SAMPLING_MODE, exp_config.SAMPLE_EVERY_N_MINUTES)
# CALL FUNCTION TO READ THE POOLED VALIDATION ROWS (SAME SAMPLING; ADJACENT QUARTERS CAN SHARE A SESSION, DUPLICATES DROPPED)
signal_valid_pdf = pd.concat([read_model_dataset_pdf(str(row["valid_start"]), str(row["valid_end"]), exp_config.SIGNAL_CHECK_DELTA_LIST, exp_config.SAMPLING_MODE,
                                                     exp_config.SAMPLE_EVERY_N_MINUTES, alert_in=False) for _, row in pooled_fold_pdf.iterrows()], ignore_index=True) \
                     .drop_duplicates(subset=["decision_ts"]).reset_index(drop=True)
# DEFINE THE FEATURE LIST
signal_scale_type_list = exp_config.MODEL_FEATURE_SCALE_TYPE_LIST + (["absolute"] if INCLUDE_ABSOLUTE_FEATURES_BOOL else [])
signal_feature_col_str_list = get_model_feature_col_str_list(signal_train_pdf, signal_scale_type_list)
# DISPLAY INFORMATION
print(f"Train Rows:\t{len(signal_train_pdf):,}\tPooled Validation Rows:\t{len(signal_valid_pdf):,}")
print(f"Features ({len(signal_feature_col_str_list)}):\t{signal_feature_col_str_list}")

In [ ]:
"""
Unconditional take profit rate per delta (no feature) = the BASE RATE every bin is compared with. Because SPY drifted
upwards, wide deltas held for days show TP rates above break-even without any signal: that is drift, not information.
"""
# LIST TO HOLD THE ROWS
unconditional_dict_list = []
# ITERATE OVER THE WINDOWS AND DELTAS
for window_str, window_pdf in [("train", signal_train_pdf), ("valid", signal_valid_pdf)]:
    for delta in exp_config.SIGNAL_CHECK_DELTA_LIST:
        # DEFINE THE LABEL COLUMNS
        delta_col_str = get_delta_col_str(delta)
        resolved_pdf = window_pdf[window_pdf[f"y_tp_{delta_col_str}"].notna()]
        # APPEND THE ROW
        unconditional_dict_list.append({"window": window_str, "delta": delta, "row_count": len(resolved_pdf),
                                        "base_tp_rate": resolved_pdf[f"y_tp_{delta_col_str}"].mean(),
                                        "breakeven_tp_rate": get_breakeven_tp_rate_arr(resolved_pdf["entry_price"].to_numpy(), delta).mean(),
                                        "mean_net_return": resolved_pdf[f"net_return_{delta_col_str}"].mean(),
                                        "tl_share": (resolved_pdf[f"exit_reason_{delta_col_str}"] == "TL").mean()})
# CONVERT TO A DATAFRAME
unconditional_pdf = pd.DataFrame(unconditional_dict_list)
# CREATE A FIGURE
fig = px.line(unconditional_pdf.melt(id_vars=["window", "delta"], value_vars=["base_tp_rate", "breakeven_tp_rate"]),
              x="delta", y="value", color="variable", line_dash="window", markers=True, log_x=True,
              title="Base (Unconditional) TP Rate vs Breakeven TP Rate", template="plotly_dark", width=1100, height=500)
# DISPLAY FIGURE
fig.show()
# PREVIEW DATAFRAME
unconditional_pdf

In [ ]:
# RECORD THE START TIME
start_time = time.time()
# CALL FUNCTION TO RUN THE SIGNAL CHECK (PROTOCOL SETTINGS FROM THE EXPERIMENT CONFIG)
signal_check_pdf = get_signal_check_pdf(signal_train_pdf, signal_valid_pdf, signal_feature_col_str_list, exp_config.SIGNAL_CHECK_DELTA_LIST)
# DISPLAY INFORMATION
print(f"Real check processed in {(time.time() - start_time) / 60:.1f} minutes")

In [ ]:
# RECORD THE START TIME
start_time = time.time()
# CALL FUNCTION TO RUN THE NULL CHECKS (FEATURES SHIFTED BY WHOLE SESSIONS; ALL CORES)
print(f"Running {exp_config.SIGNAL_CHECK_NULL_RUN_COUNT} null checks in parallel ...")
null_count_pdf = get_null_signal_count_pdf(signal_train_pdf, signal_valid_pdf, signal_feature_col_str_list, exp_config.SIGNAL_CHECK_DELTA_LIST)
# CALL FUNCTION TO GET THE VERDICT (CALIBRATED ON THE NULL)
signal_verdict_dict = get_signal_check_verdict_dict(signal_check_pdf, null_count_pdf)
# DISPLAY INFORMATION
print(f"Null checks processed in {(time.time() - start_time) / 60:.1f} minutes\n")
for key_str, value in signal_verdict_dict.items():
    print(f"{key_str}:\t{value}")
# PREVIEW THE NULL RUNS
null_count_pdf

In [ ]:
# SAVE THE SIGNAL CHECK RESULTS
write_csv_file_to_path(signal_check_pdf, f"{output_path_str}signal_check_data.csv", "W")
write_csv_file_to_path(unconditional_pdf, f"{output_path_str}base_rate_data.csv", "W")
write_csv_file_to_path(null_count_pdf, f"{output_path_str}null_run_data.csv", "W")
# LOG THE TRIAL (THE LATEST TEST WINDOW WAS NOT READ)
signal_trial_dict = log_trial_dict(exp_config, "signal_check",
                                   {"train_years": exp_config.SIGNAL_CHECK_TRAIN_WINDOW_YEARS, "train_start": train_start, "train_end": train_end,
                                    "valid_fold_id_list": pooled_fold_pdf["fold_id"].tolist(), "feature_count": len(signal_feature_col_str_list),
                                    "delta_list": exp_config.SIGNAL_CHECK_DELTA_LIST, "include_absolute_features": INCLUDE_ABSOLUTE_FEATURES_BOOL,
                                    "reference": "base rate", "min_month_count": exp_config.SIGNAL_CHECK_MIN_MONTH_COUNT,
                                    "null_run_count": exp_config.SIGNAL_CHECK_NULL_RUN_COUNT, "null_shift_share_range": exp_config.SIGNAL_CHECK_NULL_SHIFT_SHARE_RANGE},
                                   signal_verdict_dict, False, pooled_fold_pdf.iloc[-1], note_str_in="exp01 step02 signal check (stopping rule, two-sided, base rate, month minimum, null-calibrated verdict)")
print(f"Logged 1 signal-check entry (config hash {get_config_hash_str(exp_config)})")

In [ ]:
# CALCULATE THE LIFT OVER THE BASE RATE IN EACH WINDOW
signal_check_pdf["train_lift"] = signal_check_pdf["train_tp_rate"] - signal_check_pdf["train_base_rate"]
signal_check_pdf["valid_lift"] = signal_check_pdf["valid_tp_rate"] - signal_check_pdf["valid_base_rate"]
# COLLECT THE LARGEST ABSOLUTE TRAINING LIFT PER FEATURE AND DELTA (TESTED BINS)
tested_pdf = signal_check_pdf[signal_check_pdf["tested"]]
best_lift_pdf = tested_pdf.loc[tested_pdf.groupby(["feature", "delta"])["train_lift"].apply(lambda s: s.abs().idxmax())].pivot(index="feature", columns="delta", values="train_lift")
# CREATE A HEATMAP (SIGNED LIFT OF THE STRONGEST TRAINING BIN)
fig = px.imshow(best_lift_pdf, color_continuous_scale="RdYlGn", color_continuous_midpoint=0, aspect="auto",
                title="Strongest Training Lift (bin TP rate - base rate) Per Feature And Delta", template="plotly_dark", width=1100, height=900)
# DISPLAY FIGURE
fig.show()

In [ ]:
# PREVIEW THE SIGNAL BINS (SAME SIDE ON TRAINING AND POOLED VALIDATION)
display(signal_check_pdf[signal_check_pdf["signal"]].sort_values("valid_lift", ascending=False))
# PREVIEW THE TESTED BINS WITH THE LARGEST TRAINING LIFT (BOTH SIDES)
tested_pdf.reindex(tested_pdf["train_lift"].abs().sort_values(ascending=False).index)[
    ["feature", "delta", "bin", "train_row_count", "train_month_count", "train_tp_rate", "train_tp_rate_ci_low", "train_tp_rate_ci_high", "train_base_rate",
     "valid_row_count", "valid_month_count", "valid_tp_rate", "valid_base_rate", "train_breakeven_tp_rate", "signal"]].head(30).round(4)

In [ ]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)